In [24]:
import pandas as pd
data = pd.read_csv('regression_practice.csv')

data.head()

,customer_id,age,tenure_months,monthly_spend,sessions_per_week,region,premium,annual_value
0,100000,22,74,150.02,3.90,east,0,5745
1,100001,55,94,89.30,0.80,south,1,6130
2,100002,49,89,29.17,3.85,east,1,6272
3,100003,39,16,31.31,3.85,north,0,4606
4,100004,38,99,219.94,4.20,south,0,6572


In [25]:
#region kısmı kategorik olduğu için one hot encoding uygulayacağız.

encoded_data = pd.get_dummies(data,columns=['region'])
encoded_data

encoded_data=encoded_data.drop(columns=['region_west'])
encoded_data

,customer_id,age,tenure_months,monthly_spend,sessions_per_week,premium,annual_value,region_east,region_north,region_south
0,100000,22,74,150.02,3.90,0,5745,True,False,False
1,100001,55,94,89.30,0.80,1,6130,False,False,True
2,100002,49,89,29.17,3.85,1,6272,True,False,False
3,100003,39,16,31.31,3.85,0,4606,False,True,False
4,100004,38,99,219.94,4.20,0,6572,False,False,True
...,...,...,...,...,...,...,...,...,...,...
595,100595,56,76,33.69,6.75,1,6728,False,True,False
596,100596,45,86,147.19,4.41,0,6251,True,False,False
597,100597,55,118,19.95,0.58,0,5304,False,False,True
598,100598,48,97,131.92,5.91,0,7472,False,False,False


In [26]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
from sklearn.model_selection import train_test_split

y= encoded_data.iloc[:,6:7]
y

x=encoded_data.drop(columns=['annual_value','customer_id'])
x

,age,tenure_months,monthly_spend,sessions_per_week,premium,region_east,region_north,region_south
0,22,74,150.02,3.90,0,True,False,False
1,55,94,89.30,0.80,1,False,False,True
2,49,89,29.17,3.85,1,True,False,False
3,39,16,31.31,3.85,0,False,True,False
4,38,99,219.94,4.20,0,False,False,True
...,...,...,...,...,...,...,...,...
595,56,76,33.69,6.75,1,False,True,False
596,45,86,147.19,4.41,0,True,False,False
597,55,118,19.95,0.58,0,False,False,True
598,48,97,131.92,5.91,0,False,False,False


In [27]:
x_train,x_test,y_train,y_test =train_test_split(x,y,test_size=0.33,random_state=0)
lr.fit(x_train,y_train)
lr_predict=lr.predict(x_test)
lr_predict

array([[5564.40487037],
       [5675.09752791],
       [6378.01382737],
       [6693.80601393],
       [7243.14852046],
       [5958.56455278],
       [6385.35439892],
       [6465.72030017],
       [6148.75989268],
       [6339.49568058],
       [5648.30553637],
       [6329.77332629],
       [6391.98083236],
       [5228.18415167],
       [7049.71514941],
       [5801.92968407],
       [5733.21114869],
       [7128.8090659 ],
       [6271.28343926],
       [7069.18627908],
       [6084.56025135],
       [7063.51714087],
       [7470.77757981],
       [5483.53204466],
       [4988.46746898],
       [5143.98328158],
       [6409.08195215],
       [5850.06304592],
       [5453.07929489],
       [6288.50380445],
       [6905.27311128],
       [6589.40035344],
       [5963.619675  ],
       [6432.00947205],
       [5130.39722437],
       [7250.78297648],
       [7366.83867684],
       [7351.48959057],
       [5114.83426011],
       [6014.49497685],
       [6263.95697129],
       [5241.507

In [28]:
from sklearn.metrics import r2_score
r2_linear = r2_score(y_test,lr_predict)
r2_linear

0.6708673323486485

In [29]:
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(max_depth=5,random_state=42)
dt.fit(x_train,y_train)
dt_predict = dt.predict(x_test)
r2_dt = r2_score(y_test,dt_predict)
r2_dt

0.4630397661201273

In [30]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor(n_estimators=100,random_state=42)
rf.fit(x_train,y_train)
rf_predict = rf.predict(x_test)
r2_rf = r2_score(y_test,rf_predict)
r2_rf

C:\Users\Görkem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.6437730302909388

In [31]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
#SVR noktalar arası mesafe hesapladığı için elimizdeki değerleri scale ediyoruz.

sc_x=StandardScaler()
sc_y=StandardScaler()
x_train_scaled=sc_x.fit_transform(x_train)
x_test_scaled=sc_x.transform(x_test)
y_train_scaled = sc_y.fit_transform(y_train.values.reshape(-1,1)).ravel()

svr =SVR(kernel='rbf')
svr.fit(x_train_scaled,y_train_scaled)
svr_predict=sc_y.inverse_transform(svr.predict(x_test_scaled).reshape(-1,1))

r2_svr = r2_score(y_test,svr_predict)
r2_svr

0.654504485093145

In [32]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2)
x_poly_train = poly.fit_transform(x_train)
x_poly_test = poly.transform(x_test)
lr_poly=LinearRegression()
lr_poly.fit(x_poly_train,y_train)

r2_poly=r2_score(y_test,lr_poly.predict(x_poly_test))
r2_poly

0.6976922472882439

# SONUÇ
* Linear Regression -> 0.67
* Decision Tree -> 0.46
* Random Forest -> 0.64
* SVR -> 0.65
* 🏆 Polynomial Regression -> 0.69